# 04 - Business Insights & Recommendations

**E-Commerce Revenue & Customer Analytics**

This notebook synthesizes findings from notebooks 01-03 into concrete,
decision-oriented business insights: where high revenue does NOT mean
high profit, discount efficiency, shipping's effect on ratings, and a
final prioritized recommendation list.

Run `python scripts/run_pipeline.py` from the project root before executing this notebook.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROCESSED_DIR = os.path.join("..", "data", "processed")

In [ ]:
orders = pd.read_csv(os.path.join(PROCESSED_DIR, "orders_enriched.csv"), parse_dates=["order_date"])
order_items = pd.read_csv(os.path.join(PROCESSED_DIR, "order_items_enriched.csv"))
products = pd.read_csv(os.path.join(PROCESSED_DIR, "products_clean.csv"))
reviews = pd.read_csv(os.path.join(PROCESSED_DIR, "reviews_clean.csv"))
shipping = pd.read_csv(os.path.join(PROCESSED_DIR, "shipping_clean.csv"), parse_dates=["shipping_date", "delivery_date"])
rfm = pd.read_csv(os.path.join(PROCESSED_DIR, "rfm_customer_segments.csv"))
segment_summary = pd.read_csv(os.path.join(PROCESSED_DIR, "rfm_segment_summary.csv"))

valid_orders = orders[orders["order_status"] != "Cancelled"].copy()
item_orders = order_items.merge(valid_orders[["order_id"]], on="order_id", how="inner")

## 1. High revenue vs. high profit: where do they diverge?

In [ ]:
category_perf = item_orders.groupby("category").agg(
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
).reset_index()
category_perf["margin_pct"] = (category_perf["profit"] / category_perf["revenue"] * 100).round(2)
category_perf = category_perf.sort_values("revenue", ascending=False)

fig, ax1 = plt.subplots(figsize=(11, 5))
x = np.arange(len(category_perf))
ax1.bar(x, category_perf["revenue"], color="#4C72B0", label="Revenue")
ax1.set_ylabel("Revenue (Rs)", color="#4C72B0")
ax1.set_xticks(x)
ax1.set_xticklabels(category_perf["category"], rotation=45, ha="right")

ax2 = ax1.twinx()
ax2.plot(x, category_perf["margin_pct"], color="#C44E52", marker="o", linewidth=2, label="Profit Margin %")
ax2.set_ylabel("Profit Margin (%)", color="#C44E52")

ax1.set_title("Revenue vs. Profit Margin by Category", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

category_perf

**What does this tell the business?** Look for categories that rank high on the revenue bars but low on the margin line -- these generate top-line growth but contribute disproportionately little to the bottom line. That combination typically points to aggressive discounting or thin per-unit margins in that category, and is a candidate for pricing or supplier-cost renegotiation rather than a celebration of 'strong sales.'

## 2. Discount efficiency: does discounting actually grow profit?

In [ ]:
discount_bins = pd.cut(
    item_orders["discount"],
    bins=[-0.01, 0, 0.10, 0.20, 0.30, 1.0],
    labels=["0% (none)", "1-10%", "11-20%", "21-30%", "30%+"]
)
discount_perf = item_orders.groupby(discount_bins, observed=True).agg(
    line_items=("order_item_id", "count"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
).reset_index()
discount_perf.columns = ["discount_band", "line_items", "revenue", "profit"]
discount_perf["margin_pct"] = (discount_perf["profit"] / discount_perf["revenue"] * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(discount_perf["discount_band"], discount_perf["margin_pct"], color="#DD8452")
ax.set_title("Profit Margin % by Discount Band", fontsize=14, fontweight="bold")
ax.set_xlabel("Discount Band")
ax.set_ylabel("Profit Margin (%)")
plt.tight_layout()
plt.show()

discount_perf

**What does this tell the business?** Margin should decline as the discount band increases -- that's mechanically expected. The real question is whether *volume* at higher discount bands is high enough to offset the margin loss (i.e., whether discounting is driving incremental revenue or just giving away margin on sales that would have happened anyway). If the 21-30%+ bands aren't meaningfully outselling the lower bands in absolute revenue, that discounting isn't earning its keep.

## 3. Delivery time vs. customer satisfaction

In [ ]:
review_delivery = reviews.merge(orders[["order_id", "delivery_days"]], on="order_id", how="left").dropna(subset=["delivery_days"])
review_delivery["delivery_bucket"] = pd.cut(
    review_delivery["delivery_days"],
    bins=[-1, 3, 5, 7, 10, 100],
    labels=["0-3 days", "4-5 days", "6-7 days", "8-10 days", "10+ days"],
)
rating_vs_delivery = review_delivery.groupby("delivery_bucket", observed=True)["rating"].mean().round(2)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rating_vs_delivery.index.astype(str), rating_vs_delivery.values, marker="o", linewidth=2, color="#55A868")
ax.set_title("Average Rating vs. Delivery Time", fontsize=14, fontweight="bold")
ax.set_xlabel("Delivery Time Bucket")
ax.set_ylabel("Average Rating (1-5)")
ax.set_ylim(1, 5)
plt.tight_layout()
plt.show()

rating_vs_delivery

**What does this tell the business?** If average rating clearly declines as delivery time increases, shipping speed is a direct driver of customer satisfaction -- not just an operational metric in isolation. That reframes any investment in faster fulfillment (more warehouses, better carrier SLAs) as a customer-retention investment, not just a cost center.

## 4. RFM revenue concentration (Pareto check)

In [ ]:
SEGMENT_ORDER = [
    "Champions", "Loyal Customers", "Potential Loyalists", "New Customers",
    "At Risk", "Can't Lose Them", "Hibernating", "Lost",
]
segment_summary["segment"] = pd.Categorical(segment_summary["segment"], categories=SEGMENT_ORDER, ordered=True)
segment_summary = segment_summary.sort_values("segment")

champions_pct_customers = segment_summary.loc[segment_summary["segment"] == "Champions", "pct_of_customers"].values[0]
champions_pct_revenue = segment_summary.loc[segment_summary["segment"] == "Champions", "pct_of_revenue"].values[0]

top2_pct_customers = segment_summary.loc[segment_summary["segment"].isin(["Champions", "Loyal Customers"]), "pct_of_customers"].sum()
top2_pct_revenue = segment_summary.loc[segment_summary["segment"].isin(["Champions", "Loyal Customers"]), "pct_of_revenue"].sum()

print(f"Champions: {champions_pct_customers:.1f}% of customers generate {champions_pct_revenue:.1f}% of revenue")
print(f"Champions + Loyal Customers: {top2_pct_customers:.1f}% of customers generate {top2_pct_revenue:.1f}% of revenue")

**What does this tell the business?** This is the clearest evidence in the whole project for prioritizing retention spend on a small, well-defined segment rather than spreading marketing budget evenly across the customer base.

## 5. Prioritized business recommendations

The full write-up with supporting detail lives in `reports/business_recommendations.md`. Summary below:

1. **Protect Champions and Loyal Customers first.** They are a minority of the customer base but the majority of revenue -- any retention budget should go here before broad-based campaigns.
2. **Launch a targeted win-back campaign for 'At Risk' and "Can't Lose Them" segments.** These customers have proven purchase history and a track record of high spend; this is a higher-ROI channel than cold acquisition.
3. **Review discounting strategy in categories where high revenue does not translate to high profit.** Deep discounts may be driving volume without adding to the bottom line.
4. **Investigate delivery-time outliers in Standard/Economy shipping.** Rating data suggests delivery speed measurably affects customer satisfaction.
5. **Convert one-time buyers into repeat buyers.** With roughly two-thirds of active customers having ordered only once, even a modest lift in second-purchase rate has an outsized effect on total repeat-purchase revenue.
6. **Reassess pricing/margin in the highest-revenue, lowest-margin product categories.**
7. **Use CLV tiering to guide acquisition spend caps** -- the 'Low' CLV tier is where acquisition cost should be scrutinized most closely.
8. **Expand fulfillment capacity or SLAs in states with high order volume but longer average delivery times**, if such a pattern exists in the geographic data.
9. **Monitor payment failure rates by method** and address friction in the highest-failure-rate payment channel to reduce lost sales.
10. **Track New Customer segment conversion into Potential Loyalists month over month** as a leading indicator of future repeat-purchase revenue.

## Summary

This notebook translated the exploratory findings from notebooks 01-03 into specific, prioritized business actions. See `reports/executive_summary.md` and `reports/business_recommendations.md` for the polished, stakeholder-ready versions of these findings.